In [2]:
from openai import OpenAI
import re
import json
import logging
from transformers import AutoTokenizer, AutoModel
from pinecone import Pinecone
import pandas as pd

# Configure logging
logging.basicConfig(filename='../log/output.log', level=logging.INFO, format='%(message)s')
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
# 初始化 Pinecone
pc = Pinecone(api_key="40075f49-8396-4571-924a-4b6d342cc81d")

# 创建 Pinecone 索引
index_name = "arkts-defects"
dimension = 768  # BERT base 的输出维度是 768
# 连接到索引
index = pc.Index(index_name)

def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    # 使用最后一个隐藏层的平均池化作为句子嵌入
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
    return embeddings

def generate(query, system_prompt = 'You are a helpful AI assistant', base_url='https://xiaoai.plus/v1', model='gpt-4o-2024-08-06'):
    client = OpenAI(
        base_url=base_url,
        api_key="***REMOVED***"
    )

    chat_completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
    )

    return chat_completion.choices[0].message.content

rules = {
    "@performance/constant-property-referencing-check-in-loops": {
        "description": "在循环如需频繁访问某个常量，且该属性引用常量在循环中不会改变，建议提取到循环外部，减少属性访问的次数",
        "code_example": "class Time {\\n  static start: number = 0;\\n  static info: number[] = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12];\\n}\\nfunction getNum(num: number): number {\\n  /* Year has (12 * 29 =) 348 days at least */\\n  let total: number = 348;\\n  for (let index: number = 0x8000; index > 0x8; index >>= 1) {\\n    // warning\\n    total += ((Time.info[num - Time.start] & index) !== 0) ? 1 : 0;\\n  }\\n  return total;\\n}"
    },
    "@performance/foreach-args-check": {
        "description": "建议在ForEach参数中设置keyGenerator",
        "code_example": "@Entry\\n@Component\\nstruct ForeachTest {\\n  private data: string[] = ['1', '2', '3'];\\n\\n  build() {\\n    RelativeContainer() {\\n      List() {\\n        ForEach(this.data, (item: string, index: number) => {\\n          ListItem() {\\n            Text(item);\\n          }\\n        })\\n      }\\n      .width('100%')\\n      .height('100%')\\n    }\\n    .height('100%')\\n    .width('100%')\\n  }\\n}"
    },
    "@performance/high-frequency-log-check": {
        "description": "不建议在高频函数中使用Hilog。高频函数包括：onTouch、onItemDragMove、onDragMove、onMouse、onVisibleAreaChange、onAreaChange、onScroll、onActionUpdate。",
        "code_example": "// Test.ets\\nimport hilog from '@ohos.hilog';\\n@Entry\\n@Component\\nstruct Index {\\n    build() {\\n            Column() {\\n                Scroll()\\n                    .onScroll(() => {\\n                        hilog.info(1001, 'Index', 'onScroll') // Avoid printing logs\\n                })\\n            }\\n    }\\n}"
    },
    "@performance/hp-arkui-load-on-demand": {
        "description": "建议使用按需加载。",
        "code_example": "@Entry\\n@Component\\nstruct MyComponent {\\n  @State arr: number[] = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]\\n\\n  build() {\\n    List() {\\n      // List中建议使用LazyForEach\\n      ForEach(this.arr, (item: number) => {\\n        ListItem() {\\n          Text(`item value: ${item}`)\\n        }\\n      }, (item: number) => item.toString())\\n    }\\n    .width('100%')\\n    .height('100%')\\n  }\\n}"
    },
    "@performance/hp-arkui-no-state-var-access-in-loop": {
        "description": "避免在for、while等循环逻辑中频繁读取状态变量。",
        "code_example": "import hilog from '@ohos.hilog'\\n@Entry\\n@Component\\nstruct MyComponent{\\n  @State message: string = '';\\n  build() {\\n    Column() {\\n      Button('点击打印日志')\\n        .onClick(() => {\\n          this.message = 'click';\\n          for (let i = 0; i < 10; i++) {\\n            hilog.info(0x0000, 'TAG', '%{public}s', this.message);\\n          }\\n        })\\n        .width('90%')\\n        .backgroundColor(Color.Blue)\\n        .fontColor(Color.White)\\n        .margin({\\n          top: 10\\n        })\\n    }\\n    .justifyContent(FlexAlign.Start)\\n    .alignItems(HorizontalAlign.Center)\\n    .margin({\\n      top: 15\\n    })\\n  }\\n}"
    },
    "@performance/hp-arkui-no-stringify-in-lazyforeach-key-generator": {
        "description": "在使用LazyForEach进行组件复用的key生成器函数里，不要使用stringify。",
        "code_example": "import { MyDataSource } from './MyDataSource';\\n// 此处为复用的自定义组件\\n@Reusable\\n@Component\\nstruct ChildComponent {\\n  @State desc: string = '';\\n  @State sum: number = 0;\\n  @State avg: number = 0;\\n\\n  aboutToReuse(params: Record<string, Object>): void {\\n    this.desc = params.desc as string;\\n    this.sum = params.sum as number;\\n    this.avg = params.avg as number;\\n  }\\n\\n  build() {\\n    Column() {\\n      Text('子组件' + this.desc)\\n        .fontSize(30)\\n        .fontWeight(30)\\n      Text('结果' + this.sum)\\n        .fontSize(30)\\n        .fontWeight(30)\\n      Text('平均值' + this.avg)\\n        .fontSize(30)\\n        .fontWeight(30)\\n    }\\n  }\\n}\\n\\nclass Item {\\n  advertInfos: Model[] = []\\n  productPrice: PriceInfo[] = []\\n  addresses: string[] = []\\n  id: string = ''\\n}\\n\\n@Entry\\n@Component\\nstruct MyComponent {\\n  private data: MyDataSource = new MyDataSource();\\n\\n  aboutToAppear(): void {\\n    for (let index = 0; index < 20; index++) {\\n      let item = new Item()\\n      for (let i = 0; i < 1000; i++) {\\n        item.advertInfos.push(new Model(\"Product A\", \"Product A\", \"Product A\", \"Product A\", \"Product A\", \"Product A\", \"Product A\", \"Product A\"));\\n        item.productPrice.push(new PriceInfo(1.99, 123456));\\n        item.addresses.push(\"Beijing\")\\n      }\\n      item.id = index.toString();\\n      this.data.pushData(item.productPrice[0].price)\\n    }\\n  }\\n\\n  build() {\\n    Column() {\\n      Text('Use the time-consuming function `JSON.stringify (item)` to generate a key')\\n        .fontSize(12)\\n        .height('16')\\n        .margin({\\n          top: 5,\\n          bottom: 10\\n        })\\n      List() {\\n        LazyForEach(this.data, (item: Item) => {\\n          ListItem() {\\n            ChildComponent({ desc: item.id, sum: 0, avg: 0 })\\n          }\\n          .width('100%')\\n          .height('10%')\\n          .border({ width: 1 })\\n          .borderStyle(BorderStyle.Dashed)\\n        }, (item: Item) => JSON.stringify(item))\\n      }\\n      .height('100%')\\n      .width('100%')\\n    }\\n  }\\n}"
    },
    "@performance/hp-arkui-use-reusable-component": {
        "description": "建议复杂组件的定义，尽量使用组件复用。",
        "code_example": "import { MyDataSource } from './MyDataSource';\\nimport { GoodItems } from './data/DataEntry';\\n\\n@Entry\\n@Component\\nstruct MyComponent{\\n  private data: MyDataSource = new MyDataSource();\\n\\n  build() {\\n    Column() {\\n      LazyForEach(this.data, (item: GoodItems) => {\\n        GridItem() {\\n          Column() {\\n            Text(item.introduce)\\n              .fontSize(14)\\n              .padding({ left: 5, right: 5 })\\n              .margin({ top: 5 })\\n            Row() {\\n              Text('￥')\\n                .fontSize(10)\\n                .fontColor(Color.Red)\\n                .baselineOffset(-4)\\n              Text(item.price)\\n                .fontSize(16)\\n                .fontColor(Color.Red)\\n              Text(item.numb)\\n                .fontSize(10)\\n                .fontColor(Color.Gray)\\n                .baselineOffset(-4)\\n                .margin({ left: 5 })\\n            }\\n            .width('100%')\\n            .justifyContent(FlexAlign.SpaceBetween)\\n            .padding({ left: 5, right: 5 })\\n            .margin({ top: 15 })\\n          }\\n          .borderRadius(10)\\n          .backgroundColor(Color.White)\\n          .clip(true)\\n          .width('100%')\\n          .height(290)\\n        }\\n      }, (item: GoodItems) => item.index)\\n    }\\n  }\\n}"
    },
    "@performance/lottie-animation-destroy-check": {
        "description": "当使用lottie加载动画时，建议在动画完成后及时销毁以防止内存浪费。",
        "code_example": "import lottie from '@ohos.lottie';\\nimport { AnimationItem } from '@ohos.lottie';\\n\\nconst FRAME_START: number = 60;\\nconst FRAME_END: number = 120;\\n\\n@Entry\\n@Component\\nstruct LottieAnimation1 {\\n  private politeChickyController: CanvasRenderingContext2D = new CanvasRenderingContext2D();\\n  private politeChicky: string = 'politeChicky';\\n  private politeChickyPath: string = 'media/politeChicky.json';\\n  private animateItem?: AnimationItem;\\n\\n  build() {\\n    Canvas(this.politeChickyController)\\n      .width(160)\\n      .height(160)\\n      .backgroundColor(Color.Gray)\\n      .borderRadius(3)\\n      .onReady(() => {\\n        //告警\\n        this.animateItem = lottie.loadAnimation({\\n          container: this.politeChickyController,\\n          renderer: 'canvas',\\n          loop: true,\\n          autoplay: true,\\n          name: this.politeChicky,\\n          path: this.politeChickyPath,\\n          initialSegment: [FRAME_START, FRAME_END]\\n        })\\n      })\\n  }\\n}"
    },
    "@performance/multiple-associations-state-var-check": {
        "description": "多个组件关联同一数据时，建议在组件中使用@Watch装饰器添加更新条件，避免不必要的组件更新。",
        "code_example": "@Observed\\nclass UIStyle {\\n  fontSize: number = 0;\\n  fontColor: string = '';\\n  isChecked: boolean = false;\\n}\\n@Entry\\n@Component\\nstruct MultipleAssociationsStateVarReport0 {\\n  @State uiStyle: UIStyle = new UIStyle();\\n  private listData: string[] = [];\\n  aboutToAppear(): void {\\n    for (let i = 0; i < 10; i++) {\\n      this.listData.push(`ListItemComponent ${i}`);\\n    }\\n  }\\n  build() {\\n    Row() {\\n      Column() {\\n        CompA({item: '1', index: 1, subStyle: this.uiStyle})\\n        CompB({item: '2', index: 2, subStyle: this.uiStyle})\\n        CompC({item: '3', index: 3, subStyle: this.uiStyle})\\n        Text('change state var')\\n          .onClick(()=>{\\n            this.uiStyle.fontSize = 20;\\n          })\\n      }\\n      .width('100%')\\n    }\\n    .height('100%')\\n  }\\n}\\n@Component\\nstruct CompA {\\n  @Prop item: string;\\n  @Prop index: number;\\n  @Link subStyle: UIStyle;\\n  private sizeFont: number = 50;\\n  isRender(): number {\\n    console.info(`CompA ${this.index} Text is rendered`);\\n    return this.sizeFont;\\n  }\\n  build() {\\n    Column() {\\n      Text(this.item)\\n        .fontSize(this.isRender())\\n        .fontSize(this.subStyle.fontSize)\\n      Text('abc')\\n    }\\n  }\\n}\\n@Component\\nstruct CompB {\\n  @Prop item: string;\\n  @Prop index: number;\\n  @Link subStyle: UIStyle;\\n  private sizeFont: number = 50;\\n  isRender(): number {\\n    console.info(`CompB ${this.index} Text is rendered`);\\n    return this.sizeFont;\\n  }\\n  build() {\\n    Column() {\\n      Text(this.item)\\n        .fontSize(this.isRender())\\n        .fontColor(this.subStyle.fontColor)\\n      Text('abc')\\n    }\\n  }\\n}\\n@Component\\nstruct CompC {\\n  @Prop item: string;\\n  @Prop index: number;\\n  @Link subStyle: UIStyle;\\n  private sizeFont: number = 50;\\n  isRender(): number {\\n    console.info(`CompC ${this.index} Text is rendered`);\\n    return this.sizeFont;\\n  }\\n  build() {\\n    Column() {\\n      if (this.subStyle.isChecked) {\\n        Text('checked')\\n      } else {\\n        Text('unchecked')\\n      }\\n    }\\n  }\\n}"
    },
    "@performance/no-high-loaded-frame-rate-range": {
        "description": "不允许锁定最高帧率运行。",
        "code_example": "let sync: displaySync.DisplaySync = displaySync.create();\\nsync.setExpectedFrameRateRange({\\n  expected: 120,\\n  min: 120,\\n  max: 120,\\n});"
    },
    "@performance/number-init-check": {
        "description": "该规则将检查number是否正确使用。",
        "code_example": "let width = 10; width = 5.6;"
    },
    "@performance/sparse-array-check": {
        "description": "建议避免使用稀疏数组。",
        "code_example": "let count = 100000;\\nlet result: number[] = new Array(count);\\nresult = new Array();\\nresult[9999] = 0;"
    },
    "@performance/timezone-interface-check": {
        "description": "在获取非本地时间时，建议使用统一标准的i18n.Calendar接口获取时间时区相关信息。",
        "code_example": "import systemDateTime from '@ohos.systemDateTime';\\nsystemDateTime.setTimezone();"
    },
    "@performance/typed-array-check": {
        "description": "数值数组推荐使用TypedArray。",
        "code_example": "const typedArray1: number[] = new Array(1, 2, 3);\\nconst typedArray2: number[] = new Array(4, 5, 6);\\nlet res: number[] = new Array(3);\\nfor (let i = 0; i < 3; i++) {\\n     res[i] = typedArray1[i] + typedArray2[i];\\n}"
    },
    "@performance/waterflow-data-preload-check": {
        "description": "建议对waterflow子组件进行数据预加载。",
        "code_example": "WaterFlow() {\\n  LazyForEach(this.dataSource, (item: number) => {\\n    FlowItem() {\\n      ReusableFlowItem({ item: item })\\n    }\\n    .width('100%')\\n    .height(this.itemHeightArray[item % 100])\\n    .backgroundColor(this.colors[item % 5])\\n  }, (item: string) => item)\\n}\\n.onReachEnd(() => {\\n  console.info(\"onReachEnd\")\\n  setTimeout(() => {\\n    for (let i = 0; i < 100; i++) {\\n      this.datasource.AddLastItem()\\n    }\\n  }, 1000)\\n})"
    },
    "@security/no-cycle": {
        "description": "该规则禁止使用循环依赖。",
        "code_example": "// Foo.ets\\nimport {} from './Bar.ets';\\n\\n// Bar.ets\\nimport {} from './Foo.ets';"
    }
}

def get_system_prompt():
    system_prompt = ""
    # 测试输出构造好的rules字典
    for rule, details in rules.items():
        system_prompt += f"Rule: {rule}\nDescription: {details['description']}\nCode Example: {details['code_example']}\n"

    system_prompt += """我将给出代码，请判断代码中存在上述哪种缺陷类型，给出的结果需要为数组中的一项: [@performance/constant-property-referencing-check-in-loops, @performance/foreach-args-check, @performance/high-frequency-log-check, @performance/hp-arkui-load-on-demand, @performance/hp-arkui-no-state-var-access-in-loop, @performance/hp-arkui-no-stringify-in-lazyforeach-key-generator, @performance/hp-arkui-use-reusable-component, @performance/lottie-animation-destroy-check, @performance/multiple-associations-state-var-check, @performance/no-high-loaded-frame-rate-range, @performance/number-init-check, @performance/sparse-array-check, @performance/timezone-interface-check, @performance/typed-array-check, @performance/waterflow-data-preload-check, @security/no-cycle]
请以json格式输出, 不要输出任何额外信息。若存在多种缺陷，请选择最重要的缺陷进行输出，例如:
{
    "rule": "@performance/constant-property-referencing-check-in-loops",
    "description": "在循环如需频繁访问某个常量，且该属性引用常量在循环中不会改变，建议提取到循环外部，减少属性访问的次数",
    "line": 10,
    "defect snippet": "for (let i = 0; i < arr.length; i++) {\n    console.log(arr[1]);\n}",
}"""

    return system_prompt


def read_code():
    df_input = pd.read_excel('../data/test.xlsx', header=None)
    demo = df_input.iloc[1].to_dict()
    return demo[2]

def handle_vul_type_res(res):
    json_match = re.search(r'\{.*?\}', res, re.DOTALL)
    if json_match:
        json_str = json_match.group(0)

        # 尝试解析提取的 JSON 字符串
        try:
            data = json.loads(json_str)

            # 提取字段
            rule = data.get('rule')
            description = data.get('description')
            line = data.get('line')
            defect_snippet = data.get('defect snippet')

            return data

        except json.JSONDecodeError as e:
            print(f"解析 JSON 时出错: {e}")
    else:
        print("未找到 JSON 数据")

def find_vul_code(data):
    rule = data['rule']
    code = data['problem_code']
    description = data['description']
    prompt_find_vul_code = f"""
    缺陷规则如下：
    {rule}
    缺陷定义如下：
    {description}
    缺陷例子如下：
    {rules[rule]['code_example']}
    请从我的缺陷代码中寻找出哪些代码存在上述缺陷类型的问题
    缺陷代码如下：
    {code}
    """

    res = generate(prompt_find_vul_code)

    return res

def get_fix_prompt(repair_example):
    query_text = json.dumps(repair_example)
    query_vector = get_embedding(query_text)
    
    results = index.query(
        namespace="arkts",
        vector=query_vector.tolist(),
        top_k=10,
        include_metadata=True,
        filter={"rule": repair_example["rule"]}
    )
    
    fix_prompt = "下面我将给出你类似的错误，请根据这些错误的修复方案，帮我修复一下我的代码。\n"
    matches = results.matches
    for j, match in enumerate(matches):
        metadata_text = match['metadata']['text']
        try:
            parsed_text = json.loads(metadata_text)
            fix_prompt += (f"Demo {j+1}: \n问题类型规则: \n{parsed_text['rule']}\n\n问题描述: \n{parsed_text['description']}\n\n"
                       f"问题代码: \n{parsed_text['problem_code']}\n\n问题修复解释: \n{parsed_text['problem_explain']}\n\n"
                       f"修复代码: \n\n{parsed_text['problem_fix']}\n\n")
        except json.JSONDecodeError as e:
            logging.error(f"Error decoding JSON for ID {match['id']}: {e}")
            logging.error(f"Original metadata text: {metadata_text}")
            
    # print(fix_prompt)
    
    return fix_prompt

/home/miniconda3/envs/cangjieLLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`

In [3]:
# 读取代码
code = read_code()
# 读取寻找缺陷类型系统prompt
sys_prompt = get_system_prompt()

# 寻找缺陷类型
res = generate("下面是我的arkts代码: \n" + code, system_prompt=sys_prompt)
print(res)

```json
{
    "rule": "@performance/high-frequency-log-check",
    "description": "不建议在高频函数中使用Hilog。高频函数包括：onTouch、onItemDragMove、onDragMove、onMouse、onVisibleAreaChange、onAreaChange、onScroll、onActionUpdate。",
    "line": 14,
    "defect snippet": "hilog.info(1001, 'Index', 'onActionUpdate')"
}
```


In [4]:
# 处理缺陷类型结果
vul_type_data = handle_vul_type_res(res) 
vul_type_data['problem_code'] = code
# 根据缺陷类型寻找代码中的缺陷问题
vul_summary = find_vul_code(vul_type_data)
print(vul_summary)

根据缺陷规则，高频函数中不建议使用 `hilog` 进行日志打印。在你的代码中，`hilog.info` 在 `onActionUpdate` 函数中被调用，其属于高频函数，因此这个地方存在缺陷。你可以考虑移除这个日志打印或进行其他处理来优化性能。


In [6]:
# RAG来获得修复代码的prompt
fix_prompt = get_fix_prompt(vul_type_data)
print(fix_prompt)

下面我将给出你类似的错误，请根据这些错误的修复方案，帮我修复一下我的代码。
Demo 1: 
问题类型规则: 
@performance/high-frequency-log-check

问题描述: 
不建议在高频函数中使用Hilog。

高频函数包括：onTouch、onItemDragMove、onDragMove、onMouse、onVisibleAreaChange、onAreaChange、onScroll、onActionUpdate。

问题代码: 
build() {
  Stack() {
    Column() {
      LoadingPanel()
    }
    .width('100%')
    .height('100%')

    Row() {
      if (this.thumbnail !== null && this.thumbnail !== undefined) {
        Image(this.thumbnail)
          .rotate({
            x: 0,
            y: 0,
            z: 1,
            angle: 0
          })
          .onComplete((): void => {
            Log.info(TAG,
              'onComplete finish, index: ' + this.item.index + ', item: ' + JSON.stringify(this.item) + ', uri: ' +
              this.thumbnail + '.');
          })
          .onError((): void => {
            Log.error(TAG, 'image show error ' + this.thumbnail + ' ' + this.item.width + ' ' + this.item.height);
          })
      }

    }
    .width('100%')
    .height('100%'

In [7]:
# 总结修复经验
repair_suggestion = generate(fix_prompt, system_prompt="比较问题代码以及修复代码和对应的问题修复解释，总结一下该问题的修复经验")
print(repair_suggestion)

为了修复高频函数中不建议使用日志记录的问题，可以总结出以下经验，这有助于提高应用的性能：

1. **识别高频事件**：首先，了解哪些事件是高频事件，例如：`onTouch`、`onScroll`、`onDragMove` 等。

2. **避免日志记录**：在这些高频事件中，避免使用日志记录功能，如 `Log.info`，`hilog.info`，`console.log` 等，因为它们会频繁触发，导致性能问题。

3. **删除或替换日志语句**：将高频事件中的日志语句删除或用临时变量代替日志语句。这样可以避免频繁的 I/O 操作，从而优化性能。

4. **变量代替**：可以用赋值语句（例如：`const TAG = 'someEvent';`）来代替日志语句，以保持代码结构完整性。

通过应用以上经验，可以有效地进行代码优化，提高应用的响应速度和性能。


In [8]:
# 生成修复代码
fix_prompt += (f"下面开始错误的修复！\n我有如下代码：\n{vul_type_data['problem_code']}\n\n对应的问题类型是: {vul_type_data['rule']}\n\n"
               f"该问题类型的描述如下:{vul_type_data['description']}\n缺陷问题为: {vul_summary}\n缺陷修复建议为{repair_suggestion}请您帮我修复一下,输出包括问题修复解释以及修复代码\n")
res = generate(fix_prompt, system_prompt="你是arkts代码修复专家。你将获得用户给出的错误代码以及问题类型，以及对应问题类型的修复案例。请参考修复案例，根据用户给出的错误代码以及问题类型，帮助用户修复代码。", base_url='http://localhost:11434/v1', model='arktsLLM')
print(res)

根据你的描述和要求，以下是修复后的代码及其相关解释：

**问题修复解释**:
在高频函数 `onActionUpdate` 中使用 `hilog.info` 会导致频繁的日志记录，从而影响性能。为了优化性能，我们应该避免在这个高频事件中进行日志打印。取而代之的是，我们可以将日志信息赋值给一个临时变量，或者完全移除这些日志语句。

**修复代码**:
```typescript
@Entry
@Component
struct Index {
  build() {
    Column() {
      Text('PanGesture Offset:\nX: ' + 50 + '\n' + 'Y: ' + 100)
        .fontSize(28)
        .height(200)
        .width(300)
        .padding(20)
        .border({ width: 3 })
        .translate({ x: 50, y: 100, z: 0 })
        .gesture(
          PanGesture()
            .onActionUpdate((event: GestureEvent|undefined) => {
              const logInfo = 'onActionUpdate';
            })
        )
    }
  }
}
```

在这个修复代码中，我们将 `hilog.info` 替换为一个临时变量 `logInfo`，从而避免了频繁的日志记录，优化了性能。
